In [1]:
import pandas as pd
import numpy as np
import timeit
import urllib.request
import zipfile
import os

# Текст завдання: Звантажити та відкрити датасет
print("Завантаження архіву")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
csv_path = "household_power_consumption.txt" # Файл всередині архіву має розширення txt

# Завантажуємо лише якщо його ще немає
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
print("Дані готові до роботи!")

Завантаження архіву
Дані готові до роботи!


### Здійснити data cleaning

In [10]:

def clean_data(file_path):
    print("Зчитуємо та очищуємо дані...")
    # Вказуємо розділювач та символ пропущених значень
    df = pd.read_csv(file_path, sep=';', na_values=['?'], low_memory=False)
    
    # Видаляємо рядки, де є хоча б один пропуск (NaN)
    df = df.dropna()
    
    # Створюємо зручний стовпець типу datetime для фільтрації за часом
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
    
    # Створюємо категоріальний стовпець (День тижня) для останнього завдання з One Hot Encoding
    df['Day_of_Week'] = df['Datetime'].dt.day_name()
    
    return df

df = clean_data(csv_path)
print("Очищення завершено! Перші 5 рядків:")
display(df.head())

Зчитуємо та очищуємо дані...
Очищення завершено! Перші 5 рядків:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Day_of_Week
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,Saturday
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,Saturday
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,Saturday
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,Saturday
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,Saturday


### Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт

In [11]:
def task_1_power(dataframe):
    return dataframe[dataframe['Global_active_power'] > 5.0]

start_time = timeit.default_timer()
res_1 = task_1_power(df)
exec_time = timeit.default_timer() - start_time

print(f"Знайдено записів: {len(res_1)}")
print(f"Час виконання процедури: {exec_time:.5f} секунд")
display(res_1.head())

Знайдено записів: 17547
Час виконання процедури: 0.02532 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Day_of_Week
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,Saturday
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,Saturday
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,Saturday
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00,Saturday
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00,Saturday


### Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.


In [13]:
def task_2_current(dataframe):
    cond_current = dataframe['Global_intensity'].between(19.0, 20.0)
    cond_appliances = dataframe['Sub_metering_2'] > dataframe['Sub_metering_3']
    return dataframe[cond_current & cond_appliances]

start_time = timeit.default_timer()
res_2 = task_2_current(df)
exec_time = timeit.default_timer() - start_time

print(f"Знайдено записів: {len(res_2)}")
print(f"Час виконання процедури: {exec_time:.5f} секунд")
display(res_2.head())

Знайдено записів: 2509
Час виконання процедури: 0.04783 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Day_of_Week
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00,Saturday
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00,Sunday
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00,Sunday
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00,Sunday
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00,Sunday


### Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії


In [14]:
def task_3_random_mean(dataframe):
    # Випадкова вибірка без повторень (replace=False)
    sample_df = dataframe.sample(n=500000, replace=False, random_state=42)
    # Рахуємо середнє
    return sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

start_time = timeit.default_timer()
res_3 = task_3_random_mean(df)
exec_time = timeit.default_timer() - start_time

print(f"Час виконання процедури: {exec_time:.5f} секунд")
print("\nСередні величини споживання:")
print(res_3)

Час виконання процедури: 0.47760 секунд

Середні величини споживання:
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64


### Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.


In [15]:
def task_4_complex(dataframe):
    # Година >= 18 (після 18:00)
    cond_time = dataframe['Datetime'].dt.hour >= 18
    # Потужність > 6
    cond_power = dataframe['Global_active_power'] > 6.0
    # Група 2 є найбільшою
    cond_group2 = (dataframe['Sub_metering_2'] > dataframe['Sub_metering_1']) & (dataframe['Sub_metering_2'] > dataframe['Sub_metering_3'])
    
    # Фільтруємо і скидаємо індекси для правильного розрізання навпіл
    filtered = dataframe[cond_time & cond_power & cond_group2].reset_index(drop=True)
    
    # Ділимо навпіл
    half_index = len(filtered) // 2
    first_half = filtered.iloc[:half_index]
    second_half = filtered.iloc[half_index:]
    
    # Кожен третій з першої, кожен четвертий з другої (використовуємо зрізи [початок:кінець:крок])
    res_first = first_half.iloc[::3]
    res_second = second_half.iloc[::4]
    
    # З'єднуємо назад
    return pd.concat([res_first, res_second])

start_time = timeit.default_timer()
res_4 = task_4_complex(df)
exec_time = timeit.default_timer() - start_time

print(f"Знайдено записів після маніпуляцій: {len(res_4)}")
print(f"Час виконання процедури: {exec_time:.5f} секунд")
display(res_4.head())

Знайдено записів після маніпуляцій: 310
Час виконання процедури: 0.18814 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Day_of_Week
0,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00,Saturday
3,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00,Saturday
6,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00,Thursday
9,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00,Thursday
12,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00,Thursday


### Завдання: Пронормувати та стандартизувати вибраний датасет

In [6]:
def normalize_and_standardize(dataframe, col_name):
    # Нормалізація: (X - Min) / (Max - Min)
    norm = (dataframe[col_name] - dataframe[col_name].min()) / (dataframe[col_name].max() - dataframe[col_name].min())
    # Стандартизація: (X - Mean) / Std
    stand = (dataframe[col_name] - dataframe[col_name].mean()) / dataframe[col_name].std()
    return norm, stand

norm_res, stand_res = normalize_and_standardize(df, 'Global_active_power')

print("Перші 5 нормалізованих значень (від 0 до 1):")
display(norm_res.head())

print("\nПерші 5 стандартизованих значень (Z-score):")
display(stand_res.head())

Перші 5 нормалізованих значень (від 0 до 1):


0    0.374796
1    0.478363
2    0.479631
3    0.480898
4    0.325005
Name: Global_active_power, dtype: float64


Перші 5 стандартизованих значень (Z-score):


0    2.955076
1    4.037084
2    4.050325
3    4.063566
4    2.434881
Name: Global_active_power, dtype: float64

### Завдання: Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [7]:
pearson = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print("Кореляція між активною потужністю (Global_active_power) та силою струму (Global_intensity):")
print(f"Пірсон: {pearson:.4f} (дуже сильна пряма залежність)")
print(f"Спірмен: {spearman:.4f} (дуже сильна пряма залежність)")

Кореляція між активною потужністю (Global_active_power) та силою струму (Global_intensity):
Пірсон: 0.9989 (дуже сильна пряма залежність)
Спірмен: 0.9954 (дуже сильна пряма залежність)


### Завдання: Провести One Hot Encoding категоріального атрибута.

In [9]:
# Використовуємо створений раніше стовпець 'Day_of_Week' як категоріальний атрибут.
ohe_df = pd.get_dummies(df[['Day_of_Week']].head(5))

print("Результат One Hot Encoding для днів тижня:")
display(ohe_df)

Результат One Hot Encoding для днів тижня:


,Day_of_Week_Saturday
0,True
1,True
2,True
3,True
4,True
